# TCGA-LUAD Phase 6
Site-aware cohort expansion; preserved 12-sample pilot. Run sequentially.

In [ ]:
from google.colab import files
import zipfile, pathlib
uploaded=files.upload()
archive=next(x for x in uploaded if x.endswith(".zip"))
with zipfile.ZipFile(archive) as z: z.extractall("/content")
%cd /content/TCGA_LUAD_Phase6

In [ ]:
!pip -q install pandas numpy requests matplotlib seaborn pytest

## Download and validate expanded cohort
Begin with 15 per group; increase to 60 after inspecting available TSS overlap. This uses GDC public STAR gene-count outputs, not FASTQ.

In [ ]:
!python phase6.py --output /content/phase6_output --max-per-group 15 --seed 42

In [ ]:
import pandas as pd, json
print(open("/content/phase6_output/qc/readiness.json").read())
display(pd.read_csv("/content/phase6_output/qc/selected_tss_by_tissue.csv"))
display(pd.read_csv("/content/phase6_output/qc/sample_qc.csv").head())

## Install DESeq2 and run normalization
Colab package installation may take time. DGE is automatically blocked when site replication is inadequate.

In [ ]:
!apt-get -qq update && apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev >/dev/null
!Rscript -e "if (!requireNamespace('BiocManager', quietly=TRUE)) install.packages('BiocManager',repos='https://cloud.r-project.org'); BiocManager::install(c('DESeq2','pheatmap'),ask=FALSE,update=FALSE)"

In [ ]:
!Rscript phase6_deseq2.R /content/phase6_output /content/phase6_deseq2
!cat /content/phase6_deseq2/DGE_STATUS.txt

In [ ]:
import shutil
shutil.make_archive("/content/TCGA_LUAD_Phase6_Results","zip","/content",base_dir="phase6_output")
shutil.make_archive("/content/TCGA_LUAD_Phase6_DESeq2","zip","/content",base_dir="phase6_deseq2")
files.download("/content/TCGA_LUAD_Phase6_Results.zip")
files.download("/content/TCGA_LUAD_Phase6_DESeq2.zip")